# BA-FedSHAP — SoftwareX Reproducer (Google Colab)

**v1.0.1-softx** · DOI [10.5281/zenodo.20356218](https://doi.org/10.5281/zenodo.20356218)

This notebook reproduces SoftwareX Table 1 (the 15-cell COMPAS low-compute study) end-to-end on a Colab CPU runtime (~2–3 hours) or from pre-computed Zenodo raw results (~5 minutes).

**Runtime:** CPU (no GPU required). Tested on Colab free tier (Python 3.10/3.11 runtime).

**Steps:**
1. Mount Google Drive (for persistence across sessions)
2. Clone the repository
3. Set the persistence root
4. Install `uv` and create an isolated Python 3.10 environment
5. Install dependencies into that environment
6. (Optional) Download pre-computed raw results from Zenodo
7. Run all 15 cells OR skip to step 8 if using Zenodo results
8. Generate Table 1

## Cell 2 — Mount Google Drive

Mounts Drive so that experiment results persist across Colab sessions. Skip if you do not need persistence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Clone the repository

In [ ]:
import os

REPO_URL = 'https://github.com/roy-saurabh/ba_fedshap.git'
REPO_TAG = 'v1.0.1-softx'
REPO_DIR = '/content/ba_fedshap'

if not os.path.isdir(REPO_DIR):
    !git clone --branch {REPO_TAG} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f'Repository already cloned at {REPO_DIR}')

%cd {REPO_DIR}

## Cell 4 — Set persistence root

All results will be symlinked into Google Drive under this folder so they survive Colab disconnections.

In [ ]:
import os

PERSIST_ROOT = '/content/drive/MyDrive/ba_fedshap_v1_0_1_softx'
os.makedirs(PERSIST_ROOT, exist_ok=True)

# Symlink results/raw into Drive so outputs persist
raw_drive = os.path.join(PERSIST_ROOT, 'results', 'raw')
os.makedirs(raw_drive, exist_ok=True)

local_raw = '/content/ba_fedshap/results/raw'
if not os.path.islink(local_raw):
    os.makedirs(os.path.dirname(local_raw), exist_ok=True)
    if os.path.isdir(local_raw):
        import shutil
        shutil.rmtree(local_raw)
    os.symlink(raw_drive, local_raw)

print(f'Persistence root: {PERSIST_ROOT}')
print(f'Raw results will be written to: {raw_drive}')

## Cell 5 — Install uv

`uv` is used to create an isolated Python 3.10 environment with pinned dependencies, avoiding conflicts with Colab's system Python.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!uv --version

## Cell 6 — Check Python version

Python 3.12+ is not supported because `torch==2.1.0` and `flwr==1.5.0` do not install cleanly on 3.12. We create an isolated 3.10 environment.

In [ ]:
import sys
print(f'System Python: {sys.version}')
print('Will create isolated Python 3.10 environment in /content/py310')

## Cell 7 — Create Python 3.10 virtual environment

`--seed` ensures that `pip`, `setuptools`, and `wheel` are pre-installed in the environment, which is required for source builds (e.g. `torch`, `flwr`). Without `--seed`, `uv venv` creates a minimal environment without `pip`.

In [ ]:
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

!uv venv --python 3.10 --seed /content/py310
!uv pip install --python /content/py310/bin/python -r requirements.txt
print('Environment ready: /content/py310')

## Cell 8 — (Optional) Download pre-computed raw results from Zenodo

Skip this cell if you want to run all 15 cells from scratch (Cell 9).  
Running this cell downloads the archived `compas_raw_results.zip` (SHA-256: `378a6c5b...`) and extracts it, so you can jump directly to table generation in Cell 10.

In [ ]:
import hashlib, zipfile, urllib.request, os

ZENODO_URL = 'https://zenodo.org/records/20356218/files/compas_raw_results.zip'
EXPECTED_SHA256 = '378a6c5b91c2a760aacb6a88b4ebab48b69f3491468375067545f264d88186bb'
ZIP_PATH = '/content/compas_raw_results.zip'
EXTRACT_DIR = '/content/ba_fedshap/results/raw'

print('Downloading compas_raw_results.zip from Zenodo...')
urllib.request.urlretrieve(ZENODO_URL, ZIP_PATH)

sha = hashlib.sha256(open(ZIP_PATH, 'rb').read()).hexdigest()
assert sha == EXPECTED_SHA256, f'SHA-256 mismatch: {sha}'
print(f'SHA-256 verified: {sha}')

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
print(f'Extracted to {EXTRACT_DIR}')

import glob
cells = glob.glob(f'{EXTRACT_DIR}/**/full_eval.json', recursive=True)
print(f'Found {len(cells)} full_eval.json files (expected 15)')

## Cell 9 — Run all 15 experiment cells

**Skip this cell** if you downloaded the pre-computed results in Cell 8.

This runs the one-command reproducer shell script, which loops over 5 seeds × 3 α levels and writes `full_eval.json` per cell. Expected runtime: ~2–3 hours on Colab CPU.

In [ ]:
import os
os.environ['PATH'] = '/content/py310/bin:' + os.environ['PATH']

!bash scripts/run_softx_compas_lowcompute.sh

## Cell 10 — Generate Table 1

Reads all `full_eval.json` files and produces the aggregate CSV matching SoftwareX Table 1.

In [ ]:
import os
os.environ['PATH'] = '/content/py310/bin:' + os.environ['PATH']

!python scripts/make_softx_compas_table.py \
    --input results/raw/compas \
    --output results/tables/table1_compas_aggregate.csv

import pandas as pd
df = pd.read_csv('results/tables/table1_compas_aggregate.csv')
print(df.to_string(index=False))

## Cell 11 — Verify SHA-256 of raw results

Confirms the raw corpus matches the manifest archived on Zenodo.

In [ ]:
import json, hashlib
from pathlib import Path

manifest = json.loads(Path('results/manifest_sha256.json').read_text())
print('Manifest scope:', manifest.get('_scope', '')[:120], '...')

all_ok = True
for rel_path, meta in manifest.items():
    if rel_path == '_scope':
        continue
    p = Path(rel_path)
    if not p.exists():
        print(f'MISSING: {rel_path}')
        all_ok = False
        continue
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    ok = h == meta['sha256']
    print(f"{'OK' if ok else 'MISMATCH'}: {rel_path}")
    if not ok:
        all_ok = False

print('\nAll checks passed.' if all_ok else '\nSome checks failed — see above.')